# Batch ir-PAC Screening (All Trials)

This workflow iterates across NWB sessions, computes vmPFC↔hippocampus ir-PAC 
using every available trial (no subsampling), and aggregates per-pair significance 
calls into a reusable catalog for downstream notebooks or scripts.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nwb_analysis import get_subject_files
from nwb_analysis.cfc import (
    prepare_session_structures,
    compute_vmPFC_theta_phase,
    compute_hipp_gamma_envelope,
    compute_irpac,
    compute_pair_pac_presence,
    write_pac_presence_to_nwb,
)

sns.set(style="whitegrid", context="talk")
warnings.filterwarnings("ignore", category=RuntimeWarning)


In [ ]:
# Batch configuration
DATA_DIR = Path('/Users/jundazhu/SBCAT/000673')
SESSION_PATH_OVERRIDE = None   # Optional explicit NWB path
SUBJECT_FILTER = None          # e.g., '001'; None => all subjects in DATA_DIR
USE_ONLY_CORRECT = True

CONDITIONS = ['L1', 'L3']
EPOCH_ANALYZE = (0.0, 2.5)
THETA_BAND = (3.0, 7.0)
GAMMA_BAND = (70.0, 140.0)
CAR_MODE = 'hemisphere'
MIN_TRIALS_PER_COND = 20
PAC_SURR_N = 500
REPEATS_EQ = 200
LAG_GRID = np.arange(-0.150, 0.155, 0.005)
EXCLUDE_LAG_S = (0.010, 0.020)
PEAK_TO_PEAK_THRESH = 3000.0

PAC_Z_THRESHOLD = 1.64
PAC_PERCENTILE = 95.0
PAC_FDR_ALPHA = 0.05
PAC_DIRECTION_LABEL = 'hipp_to_vm'
PAC_FREQ_LABEL = f"{int(GAMMA_BAND[0])}-{int(GAMMA_BAND[1])}Hz"
PAC_POOLED_RNG_SEED = 4242


if SESSION_PATH_OVERRIDE:
    SESSION_PATHS = [Path(SESSION_PATH_OVERRIDE)]
elif DATA_DIR.exists():
    SESSION_PATHS = get_subject_files(DATA_DIR, subject_id=SUBJECT_FILTER)
else:
    SESSION_PATHS = []

print(f"Sessions discovered: {len(SESSION_PATHS)}")


In [ ]:
def build_pair_table(channel_meta: pd.DataFrame) -> pd.DataFrame:
    """Enumerate hippocampal × vmPFC channel pairs (hemisphere-matched)."""
    hipp = channel_meta[channel_meta['area'].str.contains('hipp', case=False, na=False)].copy()
    vm = channel_meta[channel_meta['area'].str.contains('vmpfc', case=False, na=False)].copy()
    hipp['hemisphere'] = hipp.get('hemisphere', 'unknown').fillna('unknown')
    vm['hemisphere'] = vm.get('hemisphere', 'unknown').fillna('unknown')

    pairs = []
    for hipp_row in hipp.itertuples():
        for vm_row in vm.itertuples():
            if hipp_row.hemisphere != 'unknown' and vm_row.hemisphere != 'unknown':
                if hipp_row.hemisphere != vm_row.hemisphere:
                    continue
            pair_id = f"hipp{hipp_row.chan_id}_vm{vm_row.chan_id}"
            pairs.append({
                'pair_id': pair_id,
                'unit_id': hipp_row.chan_id,
                'hipp_chan_id': hipp_row.chan_id,
                'unit_channel': hipp_row.chan_id,
                'hipp_hemisphere': hipp_row.hemisphere,
                'chan_id': vm_row.chan_id,
                'vm_chan_id': vm_row.chan_id,
                'vm_hemisphere': vm_row.hemisphere,
            })
    return pd.DataFrame(pairs)


def process_session(session_path: Path) -> pd.DataFrame:
    """Run the ir-PAC pipeline for one session and return per-pair presence calls."""
    session_id = session_path.stem.replace('.nwb', '')
    print(f"→ {session_id}: loading session")
    session_data = prepare_session_structures(
        session_id=session_id,
        session_path=session_path,
        conditions=CONDITIONS,
        use_only_correct=USE_ONLY_CORRECT,
    )

    channel_meta = session_data['channel_meta']
    pair_table = build_pair_table(channel_meta)
    if pair_table.empty:
        print(f"   {session_id}: no hipp×vmPFC pairs; skipping")
        return pd.DataFrame()

    theta_phase = compute_vmPFC_theta_phase(
        lfp_by_channel=session_data['lfp_by_channel'],
        channel_meta=channel_meta,
        sampling_rate=session_data['lfp_fs'],
        theta_band=THETA_BAND,
        car_mode=CAR_MODE,
        hilbert_pad_sec=1.0,
        filter_order=4,
        max_peak_to_peak=PEAK_TO_PEAK_THRESH,
    )
    gamma_env = compute_hipp_gamma_envelope(
        lfp_by_channel=session_data['lfp_by_channel'],
        channel_meta=channel_meta,
        sampling_rate=session_data['lfp_fs'],
        gamma_band=GAMMA_BAND,
        hilbert_pad_sec=1.0,
        filter_order=4,
        max_peak_to_peak=PEAK_TO_PEAK_THRESH,
    )
    if not theta_phase or not gamma_env:
        print(f"   {session_id}: missing theta or gamma signals; skipping")
        return pd.DataFrame()

    pac_results = compute_irpac(
        pair_table=pair_table,
        trial_table=session_data['trial_table'],
        theta_phase=theta_phase,
        gamma_envelope=gamma_env,
        lfp_fs=session_data['lfp_fs'],
        lfp_start_time=session_data['lfp_start_time'],
        epoch_analyze=EPOCH_ANALYZE,
        conditions=CONDITIONS,
        min_trials=MIN_TRIALS_PER_COND,
        n_surrogates=PAC_SURR_N,
        repeats_eq=REPEATS_EQ,
        lag_grid_s=LAG_GRID,
        exclude_lag_s=EXCLUDE_LAG_S,
    )

    pac_presence = compute_pair_pac_presence(
        pac_results=pac_results,
        conditions=CONDITIONS,
        min_trials=MIN_TRIALS_PER_COND,
        n_surrogates=PAC_SURR_N,
        direction_label=PAC_DIRECTION_LABEL,
        freq_label=PAC_FREQ_LABEL,
        z_threshold=PAC_Z_THRESHOLD,
        percentile=PAC_PERCENTILE,
        fdr_alpha=PAC_FDR_ALPHA,
        rng_seed=PAC_POOLED_RNG_SEED,
    )
    if pac_presence.empty:
        print(f"   {session_id}: no qualifying trials/pairs")
        return pd.DataFrame()

    pac_presence.insert(0, 'session_id', session_id)
    pac_presence['session_path'] = str(session_path)
    try:
        write_pac_presence_to_nwb(session_path=session_path, pac_presence=pac_presence)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"   {session_id}: failed to embed PAC presence → {exc}")
    print(f"   {session_id}: {pac_presence['is_pac_positive_z'].sum()} PAC+ pairs (z>={PAC_Z_THRESHOLD})")
    return pac_presence




In [ ]:
batch_results = []
failures = []

for session_path in SESSION_PATHS:
    session_id = session_path.stem.replace('.nwb', '')
    try:
        df = process_session(Path(session_path))
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"{session_id}: {exc}")
        failures.append({'session_id': session_id, 'error': str(exc)})
        continue
    if df is None or df.empty:
        continue
    batch_results.append(df)

catalog = pd.concat(batch_results, ignore_index=True) if batch_results else pd.DataFrame()
print(f"Aggregated rows: {len(catalog)}")

catalog.head()


In [ ]:
if not catalog.empty:
    print('All PAC presence tables embedded directly in their NWB files (no CSV catalog written).')
else:
    print('No PAC presence data were generated; nothing embedded.')


In [ ]:
if not catalog.empty:
    display(catalog[['session_id', 'pair_id', 'z_pooled', 'is_pac_positive_z', 'fdr_significant']].head())
    per_session = (
        catalog.groupby('session_id')
        .agg(
            n_pairs=('pair_id', 'count'),
            n_used_pairs=('pair_id', pd.Series.nunique),
            n_pac_z=('is_pac_positive_z', 'sum'),
            n_pac_fdr=('fdr_significant', 'sum'),
        )
        .reset_index()
    )
    display(per_session)
else:
    print('Catalog is empty; no QC summary to show.')


In [ ]:
if not catalog.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(catalog['z_pooled'].dropna(), bins=40, color='#1f77b4')
    plt.axvline(PAC_Z_THRESHOLD, color='red', linestyle='--', label=f'z={PAC_Z_THRESHOLD}')
    plt.xlabel('Pooled ir-PAC z-score')
    plt.ylabel('Count')
    plt.title('Distribution of pooled ir-PAC z-scores')
    plt.legend()
    plt.show()
else:
    print('Catalog is empty; skipping histogram.')


In [ ]:
if failures:
    print('Failures encountered:')
    display(pd.DataFrame(failures))
else:
    print('All sessions processed successfully.')
